In [1]:
import polars as pl
import geopandas as gpd

In [2]:
df_inicial = pl.read_ods('tabela10301.ods')

In [3]:
df_inicial.head()

"Brasil, Grande Região e Unidade da Federação",2022
str,f64
"""Brasil""",0.542
"""Norte""",0.545
"""Nordeste""",0.541
"""Sudeste""",0.53
"""Sul""",0.476


In [4]:
df_inicial = (
    df_inicial.rename({
        'Brasil, Grande Região e Unidade da Federação': 'regiao',
        '2022': 'gini'
    })
)

In [5]:
df_inicial.head()

regiao,gini
str,f64
"""Brasil""",0.542
"""Norte""",0.545
"""Nordeste""",0.541
"""Sudeste""",0.53
"""Sul""",0.476


In [6]:
df_regioes = (
    df_inicial.filter(
        pl.col('regiao').is_in(['Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-Oeste'])
    )
    .sort('gini', descending=True)
)

In [7]:
df_regioes.head()

regiao,gini
str,f64
"""Norte""",0.545
"""Nordeste""",0.541
"""Centro-Oeste""",0.531
"""Sudeste""",0.53
"""Sul""",0.476


In [8]:
(
    df_regioes.rename({
        'regiao': 'Grande Região',
        'gini': 'Gini'
    })
    .to_pandas()
    .to_latex('tabela10301_regioes.tex', index=False)
)

In [9]:
df_ufs = (
    df_inicial.filter(
        ~pl.col('regiao').is_in(['Centro-Oeste', 'Sul', 'Sudeste', 'Norte', 'Nordeste', 'Brasil'])
    )
)

In [10]:
df_ufs.head()

regiao,gini
str,f64
"""Rondônia""",0.489
"""Acre""",0.547
"""Amazonas""",0.572
"""Roraima""",0.572
"""Pará""",0.539


In [11]:
gdf_ufs = gpd.read_file('../../../complementar/BR_UF_2025')

In [12]:
gdf_ufs.head()

,CD_UF,NM_UF,SIGLA_UF,CD_REGIAO,NM_REGIAO,SIGLA_RG,AREA_KM2,geometry
0,43,Rio Grande do Sul,RS,4,Sul,S,281707.150,"MULTIPOLYGON (((-53.52154 -33.25881, -53.51825..."
1,35,São Paulo,SP,3,Sudeste,SE,248219.485,"MULTIPOLYGON (((-48.03575 -25.35712, -48.03607..."
2,32,Espírito Santo,ES,3,Sudeste,SE,46074.440,"MULTIPOLYGON (((-40.88385 -21.16198, -40.88384..."
3,33,Rio de Janeiro,RJ,3,Sudeste,SE,43750.424,"MULTIPOLYGON (((-44.72025 -23.35934, -44.72029..."
4,41,Paraná,PR,4,Sul,S,199293.571,"MULTIPOLYGON (((-48.40723 -25.84254, -48.40732..."


In [16]:
gdf_temp = (
    gdf_ufs[['NM_UF', 'geometry']]
        .merge(df_ufs.to_pandas(), left_on='NM_UF', right_on='regiao')
        .drop(columns=['NM_UF'])
        .rename(columns={
            'regiao': 'uf'
        })
)

In [17]:
gdf_temp.head()

,geometry,uf,gini
0,"MULTIPOLYGON (((-53.52154 -33.25881, -53.51825...",Rio Grande do Sul,0.484
1,"MULTIPOLYGON (((-48.03575 -25.35712, -48.03607...",São Paulo,0.525
2,"MULTIPOLYGON (((-40.88385 -21.16198, -40.88384...",Espírito Santo,0.505
3,"MULTIPOLYGON (((-44.72025 -23.35934, -44.72029...",Rio de Janeiro,0.574
4,"MULTIPOLYGON (((-48.40723 -25.84254, -48.40732...",Paraná,0.482


In [18]:
gdf_temp.to_file('tabela10301_ufs.gpkg', driver='GPKG')